> 在线讲义：[第 38 章 · 从超节点走向集群：出口、割集与 Clos 网络](https://wikieden.github.io/ai-infra-from-constraints/lessons/38-cluster.html) · 本书《从约束推导 AI Infra》

# 第 38 章 · 从超节点走向集群：出口、割集与 Clos 网络

配套讲义：仓库里 `ai-infra/notes/lessons/38-cluster.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



训练状态容量、例题 7.1 的强扩展、Clos 网络的端点数与半分带宽、超售比取自 AII 第 7 章 §7.1。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
MiB = 2**20

**函数 `strong_scaling`**：(串行, 完全重叠) 的完成时间。

In [ ]:
def strong_scaling(x: float, comp_ms: float = 20, comm_ms: float = 360 * MiB / 50e9 * 1e3) -> tuple[float, float]:
    """(串行, 完全重叠) 的完成时间。"""
    c = comp_ms / x
    return c + comm_ms, max(c, comm_ms)

**函数 `clos`**：无阻塞 Clos：(端点数, 交换机数)。

In [ ]:
def clos(k: int, layers: int) -> tuple[int, int]:
    """无阻塞 Clos：(端点数, 交换机数)。"""
    if layers == 2:
        return k * k // 2, k + k // 2
    return k**3 // 4, 5 * k * k // 4

**函数 `bisection_tbps`**：小工具。

In [ ]:
def bisection_tbps(endpoints: int, port_gbps: float = 50) -> float:
    return endpoints / 2 * port_gbps / 1e3

**函数 `oversub_two_layer`**：超售两层：(端点数, 交换机数, 半分带宽 TB/s)。

In [ ]:
def oversub_two_layer(k: int = 64, down: int = 48, up: int = 16) -> tuple[int, int, float]:
    """超售两层：(端点数, 交换机数, 半分带宽 TB/s)。"""
    leaves = k
    spines = leaves * up // k
    return leaves * down, leaves + spines, leaves * up / 2 * 50 / 1e3

**函数 `job_cut`**：一个作业占整数台叶交换机：(叶数, 割集链路, 割集 TB/s, 每卡 GB/s)。

In [ ]:
def job_cut(cards: int, down: int, up: int, port_gbps: float = 50) -> tuple[int, int, float, float]:
    """一个作业占整数台叶交换机：(叶数, 割集链路, 割集 TB/s, 每卡 GB/s)。"""
    leaves = math.ceil(cards / down)
    links = leaves * up
    return leaves, links, links * port_gbps / 1e3, links * port_gbps / cards

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：容量")
print(f"  8B 混合精度 Adam 16 B/参数：{8e9 * 16 / 1e9:.0f} GB；284B：{284e9 * 16 / 1e12:.2f} TB")
print(f"  推理 0.5 B/参数：1.6T {1.6e12 * 0.5 / 1e9:.0f} GB → {math.ceil(800 / 640)} 台；2.8T {2.8e12 * 0.5 / 1e12:.1f} TB → {math.ceil(1400 / 640)} 台")

In [ ]:
print("\n第二步：例题 7.1，跨服务器 360 MiB 走一张 50 GB/s 网卡")
t = 360 * MiB / 50e9 * 1e3
for x in (1, 2, 4):
    s, o = strong_scaling(x)
    print(f"  卡数 ×{x}：串行 {s:.1f} ms，重叠 {o:.1f} ms")
print(f"  通信 {t:.2f} ms；计算 = 通信时 x = {20 / t:.2f}")

In [ ]:
print("\n第三步：Clos，k = 64")
for L in (2, 3):
    e, s = clos(64, L)
    print(f"  {L} 层无阻塞：{e:,} 个端点、{s:,} 台交换机，半分 {bisection_tbps(e):,.1f} TB/s")
e, s, b = oversub_two_layer()
print(f"  3:1 超售两层：{e} 个端点、{s} 台交换机，半分 {b} TB/s（无阻塞同规模 {bisection_tbps(e):.1f}）")
for name, d, u in (("1:1", 32, 32), ("3:1", 48, 16)):
    print(f"  1024 卡 {name}：{job_cut(1024, d, u)}")

## 3. 代码里才看得到的

1. 360 MiB ÷ 50 GB/s 是 7.55 ms，原文取 7.5 ms，交点 2.65 写作约 2.6。
2. `job_cut` 用上取整：1024 ÷ 48 = 21.3，要占 22 台叶交换机，所以 3:1 时每卡份额是 17.2 而不是 16.7 GB/s。
3. 三层交换机数 $5k^2/4$ = $k^2$（边缘 + 汇聚，每个 pod $k$ 台）+ $k^2/4$（核心）。

## 4. 自己改一改

- 让 `strong_scaling` 的通信也随卡数线性增长（弱扩展），找最优倍数。
- 用 `job_cut` 算 512、2048 卡在 2:1 超售下的每卡份额。
- 给 `clos` 加一个超售比参数。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 书包装不下才分装，分了就要来回翻；
> 一座桥过货有上限，人再多也快不过桥；
> 支路宽主干窄是超售，留在片区里就不堵；
> 下单界面统一，外省快递还是走干线。